# MNIST Digit Classification — Vision Transformer

A compact Vision Transformer (ViT) for handwritten digit classification on the [MNIST dataset](http://yann.lecun.com/exdb/mnist/): images are split into small patches, linearly embedded, and processed by a Transformer encoder with a learned `[CLS]` token used for classification.

**Result: 99.29% accuracy on the MNIST test set** (10,000 images), with a model of ~557K trainable parameters.

This notebook can be used in two ways:
- **Load the pretrained weights** included in this repository (`models/mnist_transformer_model.pth`) and go straight to evaluation — this is the default.
- **Train the model from scratch** by setting `TRAIN_FROM_SCRATCH = True` in the configuration cell below (roughly 30-45 minutes on a GPU for the full run, with early stopping).


In [ ]:
import os
import random
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [ ]:
# Reproducibility & device configuration
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Configuration

Set `TRAIN_FROM_SCRATCH` to `True` to retrain the model instead of loading the pretrained checkpoint. If no checkpoint is found at `PRETRAINED_MODEL_PATH`, the notebook automatically falls back to training from scratch.


In [ ]:
# --- Toggle: load pretrained weights vs. train from scratch ---
TRAIN_FROM_SCRATCH = False

PRETRAINED_MODEL_PATH = "models/mnist_transformer_model.pth"

SHOULD_TRAIN = TRAIN_FROM_SCRATCH or not os.path.exists(PRETRAINED_MODEL_PATH)
print(f"Mode: {'training from scratch' if SHOULD_TRAIN else 'loading pretrained weights'}")


## Dataset

MNIST is downloaded automatically via `torchvision.datasets.MNIST` on first run. The 60,000 training images are split into 55,000 for training and 5,000 for validation. Training images go through light data augmentation (random rotation, padding + random crop); validation/test images are only normalized.


In [ ]:
# Dataset
# MNIST returns PIL images by default; transforms are applied below.
full_train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=None)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=None)

# Data augmentation (training) and preprocessing (train/val/test) pipelines
train_transform_augmented = transforms.Compose([
    transforms.RandomRotation(degrees=15),                # random rotation, -15 to +15 degrees
    transforms.Pad(padding=4),                             # pad by 4 pixels on each side
    transforms.RandomCrop(size=28),                        # randomly crop back to 28x28
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),   # MNIST mean/std
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])


class DatasetWithTransform(torch.utils.data.Dataset):
    """Wraps a base dataset and applies a torchvision transform lazily, on __getitem__."""

    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, i):
        img, label = self.dataset[i]
        if self.transform:
            img = self.transform(img)
        return img, label


In [ ]:
# Train / validation split
train_subset, val_subset = random_split(
    full_train_dataset,
    lengths=[55_000, 5_000],
    generator=torch.Generator().manual_seed(SEED),
)

train_dataset = DatasetWithTransform(train_subset, train_transform_augmented)
val_dataset = DatasetWithTransform(val_subset, val_test_transform)
test_dataset = DatasetWithTransform(test_dataset, val_test_transform)


In [ ]:
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=device.type == "cuda")
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")


## Model architecture — MNISTTransformer

A small Vision Transformer (ViT-style) classifier:

- **Patch embedding:** the 28x28 image is split into non-overlapping `2x2` patches (196 patches total), each linearly projected to a 128-dimensional embedding via a strided `Conv2d`.
- A learned **`[CLS]` token** is prepended to the patch sequence, and learned **positional embeddings** are added to all 197 tokens.
- **Transformer encoder:** 4 encoder layers, each with 4 attention heads, a 256-dim feed-forward block, GELU activation, pre-layer-normalization (`norm_first=True`), and dropout 0.05.
- **Classification head:** the final `[CLS]` token representation is layer-normalized and passed through a `Linear(128 → 10)` layer.
- Weights are Xavier-initialized, biases are zero-initialized, and the `[CLS]` token / positional embeddings use truncated-normal initialization.

**Total trainable parameters: 557,450**


In [ ]:
class MNISTTransformer(nn.Module):
    def __init__(
        self,
        image_size=28,
        patch_size=4,
        in_channels=1,
        num_classes=10,
        embedding_dim=128,
        num_heads=4,
        num_layers=4,
        feedforward_dim=256,
        dropout=0.1,
    ):
        super().__init__()

        if image_size % patch_size != 0:
            raise ValueError("image_size must be divisible by patch_size")

        self.patch_size = patch_size
        patches_per_side = image_size // patch_size
        self.num_patches = patches_per_side ** 2

        # Projects each patch into an `embedding_dim`-dimensional vector.
        self.patch_embedding = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embedding_dim,
            kernel_size=patch_size,
            stride=patch_size,  # stride == kernel_size ensures non-overlapping patches
        )

        # [CLS] token used to classify the whole image (global representation).
        self.class_token = nn.Parameter(torch.zeros(1, 1, embedding_dim))

        # Positional embedding for each of the `num_patches` patches + the [CLS] token.
        self.position_embedding = nn.Parameter(torch.zeros(1, self.num_patches + 1, embedding_dim))

        self.embedding_dropout = nn.Dropout(dropout)

        # Transformer encoder layer definition
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,   # batch dimension comes first: (batch, seq_len, features)
            norm_first=True,    # apply layer norm before attention and feed-forward
        )

        # Stack `num_layers` encoder layers
        self.transformer = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=num_layers)

        # Final normalization and classification layers
        self.normalization = nn.LayerNorm(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

        self._initialize_parameters()

    def _initialize_parameters(self):
        # Truncated-normal initialization for the [CLS] token and positional embeddings
        nn.init.trunc_normal_(self.class_token, std=0.02)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

        # Initialize the remaining Transformer layers separately (except class_token / position_embedding)
        for name, parameter in self.named_parameters():
            if name in {"class_token", "position_embedding"}:
                continue
            if parameter.dim() > 1:            # weight matrices
                nn.init.xavier_uniform_(parameter)
            elif name.endswith("bias"):         # biases
                nn.init.zeros_(parameter)

    def forward(self, x):
        batch_size = x.shape[0]                     # input: [batch, 1, 28, 28]
        x = self.patch_embedding(x)                  # after conv: [batch, embedding_dim, 14, 14]
        x = x.flatten(start_dim=2)
        x = x.transpose(1, 2)                         # after flatten: [batch, num_patches, embedding_dim]

        # Expand the [CLS] token to match the batch size
        class_tokens = self.class_token.expand(batch_size, -1, -1)
        x = torch.cat([class_tokens, x], dim=1)

        # Add positional embeddings and dropout
        x = x + self.position_embedding
        x = self.embedding_dropout(x)

        # Pass through the Transformer encoder
        x = self.transformer(x)

        # Keep only the [CLS] token, which holds the global image representation
        class_representation = x[:, 0]
        class_representation = self.normalization(class_representation)

        # Return 10 logits (no softmax — the loss function handles it)
        logits = self.classifier(class_representation)

        return logits


In [ ]:
MODEL_CONFIG = dict(
    embedding_dim=128,
    num_heads=4,
    num_layers=4,
    feedforward_dim=256,
    dropout=0.05,
)

if not SHOULD_TRAIN:
    state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location="cpu", weights_only=True)

    # The patch size is encoded in the shape of the patch-embedding kernel
    # (patch_embedding.weight: [embedding_dim, in_channels, kernel_h, kernel_w]),
    # so it can be recovered directly from the checkpoint.
    patch_size = state_dict["patch_embedding.weight"].shape[-1] if "patch_embedding.weight" in state_dict else 2

    model = MNISTTransformer(patch_size=patch_size, **MODEL_CONFIG)
    model.load_state_dict(state_dict)
    model = model.to(device)
    print(f"Loaded pretrained weights from {PRETRAINED_MODEL_PATH} (patch_size={patch_size})")
else:
    patch_size = 2  # patch size used to train the released model (14x14 = 196 patches)
    model = MNISTTransformer(patch_size=patch_size, **MODEL_CONFIG).to(device)
    print("Training a new model from scratch." if TRAIN_FROM_SCRATCH else
          f"No pretrained weights found at {PRETRAINED_MODEL_PATH} — training from scratch.")


In [ ]:
number_of_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(f"Trainable parameters: {number_of_parameters:,}")


## Training configuration

- **Loss:** cross-entropy with label smoothing (0.1)
- **Optimizer:** AdamW, lr=6e-4, weight_decay=1e-2
- **Scheduler:** `ReduceLROnPlateau` on validation accuracy (factor 0.5, patience 2)
- **Epochs:** up to 100, with early stopping (patience 10, min_delta 1e-4)
- **Batch size:** 128
- **Gradient clipping:** max norm 1.0


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=6e-4,
    weight_decay=1e-2,
)

# Training hyperparameters
epochs = 100
patience = 10
min_delta = 0.0001

# Learning-rate scheduler, driven by validation accuracy
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)


In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch_idx, (images, labels) in enumerate(data_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()

        # Prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_examples += batch_size

        if batch_idx % 100 == 0:
            print(
                f"  Batch {batch_idx:04d}/{len(data_loader):04d} | "
                f"Loss: {loss.item():.4f} | "
                f"Accuracy: {(total_correct / total_examples) * 100:.2f}%"
            )

    average_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return average_loss, accuracy


In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_examples += batch_size

    average_loss = total_loss / total_examples
    accuracy = total_correct / total_examples
    return average_loss, accuracy


In [ ]:
CHECKPOINT_PATH = "mnist_transformer_best.pt"

if SHOULD_TRAIN:
    best_validation_accuracy = 0.0
    no_improve_epochs = 0

    for epoch in range(epochs):
        print(f"\n{'='*20} Epoch {epoch + 1}/{epochs} {'='*20}")

        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        validation_loss, validation_accuracy = evaluate(model, val_loader, criterion, device)

        scheduler.step(validation_accuracy)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | Train accuracy: {train_accuracy * 100:.2f}% | "
            f"Val loss: {validation_loss:.4f} | Val accuracy: {validation_accuracy * 100:.2f}% | "
            f"LR: {current_lr:.6f}"
        )

        if validation_accuracy > best_validation_accuracy + min_delta:
            best_validation_accuracy = validation_accuracy
            no_improve_epochs = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print("  New best model saved.")
        else:
            no_improve_epochs += 1
            print(f"  No significant improvement ({no_improve_epochs}/{patience})")

        if no_improve_epochs >= patience:
            print("  Early stopping triggered.")
            break
else:
    print("Skipping training — using pretrained weights. Set TRAIN_FROM_SCRATCH = True above to retrain.")


## Evaluation

In [ ]:
# If we just trained, reload the best checkpoint saved during training
if SHOULD_TRAIN:
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))

test_loss, test_accuracy = evaluate(model=model, data_loader=test_loader, criterion=criterion, device=device)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy * 100:.2f}%")


In [ ]:
# Display a random sample of predictions (mostly correct, a few incorrect)
model.eval()

correct_samples = []
incorrect_samples = []

print("Collecting predictions over the test set...")

with torch.no_grad():
    for images_batch, labels_batch in test_loader:
        images_device = images_batch.to(device)
        logits = model(images_device)
        predictions_batch = logits.argmax(dim=1).cpu()

        for i in range(len(images_batch)):
            image = images_batch[i].cpu()
            label = labels_batch[i].cpu()
            prediction = predictions_batch[i]

            if prediction == label:
                correct_samples.append((image, label, prediction))
            else:
                incorrect_samples.append((image, label, prediction))

display_samples = []

num_to_get_correct = min(8, len(correct_samples))
display_samples.extend(random.sample(correct_samples, num_to_get_correct))

num_to_get_incorrect = min(4, len(incorrect_samples))
display_samples.extend(random.sample(incorrect_samples, num_to_get_incorrect))

num_needed = 12 - len(display_samples)
if num_needed > 0:
    remaining_pool = [s for s in (correct_samples + incorrect_samples) if s not in display_samples]
    if len(remaining_pool) >= num_needed:
        display_samples.extend(random.sample(remaining_pool, num_needed))
    else:
        display_samples.extend(remaining_pool)

random.shuffle(display_samples)
display_samples = display_samples[:12]

figure = plt.figure(figsize=(12, 6))
for index, (image, label, prediction) in enumerate(display_samples):
    ax = figure.add_subplot(3, 4, index + 1)
    image = image.squeeze(0)
    image = image * 0.3081 + 0.1307  # de-normalize
    ax.imshow(image, cmap="gray")
    color = "green" if prediction == label else "red"
    ax.set_title(f"True: {label.item()} | Predicted: {prediction.item()}", color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect misclassified examples in more detail
model.eval()
all_errors = []

print("Scanning the test set for errors...")

with torch.no_grad():
    for images, labels in test_loader:
        images_dev = images.to(device)
        logits = model(images_dev)
        probs = F.softmax(logits, dim=1)

        preds = probs.argmax(dim=1).cpu()
        probs = probs.cpu()

        for i in range(len(images)):
            if preds[i] != labels[i]:
                all_errors.append({
                    "image": images[i],
                    "true_label": labels[i].item(),
                    "pred_label": preds[i].item(),
                    "true_prob": probs[i][labels[i]].item(),
                    "pred_prob": probs[i][preds[i]].item(),
                })

num_to_display = min(len(all_errors), 20)
selected_errors = all_errors[:num_to_display]

print(f"Number of errors found: {len(all_errors)}")
print(f"Displaying the first {num_to_display} errors.")

if num_to_display > 0:
    rows = (num_to_display + 3) // 4
    fig = plt.figure(figsize=(16, 4 * rows))
    for i, error in enumerate(selected_errors):
        ax = fig.add_subplot(rows, 4, i + 1)
        img = error["image"].squeeze().numpy()
        img = img * 0.3081 + 0.1307  # de-normalize

        ax.imshow(img, cmap="gray")
        ax.set_title(
            f"True: {error['true_label']} ({error['true_prob']:.2%})\n"
            f"Predicted: {error['pred_label']} ({error['pred_prob']:.2%})",
            color="red", fontsize=10,
        )
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No errors found on the test set!")


## Results

| Metric | Value |
|---|---|
| Test accuracy | **99.29%** |
| Test loss | 0.5180 |
| Trainable parameters | 557,450 |
| Patch size | 2x2 (196 patches) |

These results were obtained with the pretrained checkpoint (`models/mnist_transformer_model.pth`) included in this repository.


In [ ]:
if SHOULD_TRAIN:
    os.makedirs("models", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = f"models/mnist_transformer_model_{timestamp}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")
